In [0]:
import requests
import json
import os
from datetime import date

import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DoubleType, DateType, TimestampType

catalog_name = "ecommerce"

In [0]:
df_gold_order_items = spark.read.table(f"{catalog_name}.silver.slv_order_items_clean")

df_gold_order_items.limit(10).display()

# I. Adding New Columns

In [0]:
# 1.1 New Column: gross_amount = (unit_price * quantity)

df_gold_order_items = df_gold_order_items.withColumn(
  "gross_amount",
  F.round(F.col("unit_price") * F.col("quantity"), 2)
)

In [0]:
# 1.2 New Column: discount_amount = (gross_amount * discount_pct / 100)

df_gold_order_items = df_gold_order_items.withColumn(
    "discount_amount",
    F.round(F.col("gross_amount") * (F.col("discount_pct") / 100.0), 2)
)

In [0]:
# 1.3 New Column: net_amount = (gross_amount - discount_amount)

df_gold_order_items = df_gold_order_items.withColumn(
    "net_amount",
    F.round(F.col("gross_amount") - F.col("discount_amount"), 2)
)

In [0]:
# 1.4 New Column: final_amount = (net_amount + tax_amount)

df_gold_order_items = df_gold_order_items.withColumn(
    "final_amount",
    F.round(F.col("net_amount") + F.col("tax_amount"), 2)
)

In [0]:
# 1.5 New Column: coupon_flag = (coupon_code != null)

df_gold_order_items = df_gold_order_items.withColumn(
    "coupon_flag",
    F.when(F.col("coupon_code").isNotNull(), F.lit(1))
    .otherwise(F.lit(0))
)

In [0]:
df_gold_order_items.limit(10).display()

# II. Currency Conversion with API

In [0]:
# Fetching live rates

response = requests.get("https://api.exchangerate-api.com/v4/latest/PHP")
data_response = response.json()
rates = data_response["rates"]
rate_date = data_response["date"] #when these rates are from

print(f"FX rates as of: {rate_date}")

# 2. Build rates DataFrame
company_currencies = ["USD", "GBP", "SGD", "AUD", "AED", "PHP", "INR"]
rates_list = [(curr, float(rates.get(curr, 1.0)))
              for curr in company_currencies]

rates_df = spark.createDataFrame(rates_list, ["unit_price_currency", "fx_rate"])

#3. Join to Fact table and convert

df_gold_order_items = df_gold_order_items.join(
  rates_df, on = "unit_price_currency",
  how = "left"
).withColumn(
  "gross_amount_php",
  F.round(F.col("gross_amount") / F.col("fx_rate"), 2)
).withColumn(
  "discount_amount_php",
  F.round(F.col("discount_amount") / F.col("fx_rate"), 2)
).withColumn(
  "net_amount_php",
  F.round(F.col("net_amount") / F.col("fx_rate"), 2)
).withColumn(
  "tax_amount_php",
  F.round(F.col("tax_amount") / F.col("fx_rate"), 2)
).withColumn(
  "final_amount_php",
  F.round(F.col("final_amount") / F.col("fx_rate"), 2)
).withColumn(
  "fx_rate_date", F.lit(rate_date)
)


In [0]:
df_gold_order_items.limit(10).display()

In [0]:
df_gold_order_items = df_gold_order_items.select(
    F.col("order_date"),
    F.col("order_timestamp"),
    F.col("order_id"),
    F.col("customer_id"),
    F.col("item_seq").alias("sequence_no"),
    F.col("product_id"),
    F.col("channel"),
    F.col("coupon_code"),
    F.col("coupon_flag"),
    F.col("unit_price_currency"),
    F.col("quantity"),
    F.col("unit_price"),
    F.col("gross_amount"),
    F.col("discount_pct").alias("discount_percent"),
    F.col("discount_amount"),
    F.col("net_amount"),
    F.col("tax_amount"),
    F.col("final_amount"),
    F.col("fx_rate_date"),
    F.col("gross_amount_php"),
    F.col("discount_amount_php"),
    F.col("net_amount_php"),
    F.col("tax_amount_php"),
    F.col("final_amount_php"),
    F.col("file_name"),
    F.col("ingest_timestamp"),
    F.col("processed_time")
)

In [0]:
df_gold_order_items.limit(10).display()

# III. Write to Delta

In [0]:
df_gold_order_items.write.format(
  "delta"
).mode(
  "overwrite"
).saveAsTable(
  f"{catalog_name}.gold.gld_fact_order_items"
  )
